<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

# Subconsultas (subselects)
Un `SELECT` dentro de otro `SELECT`. Según dónde lo ponga se comporta distinto:

| Dónde | Para qué se usa |
|---|---|
| En el `FROM` | Se trata como una tabla (tabla derivada), **siempre lleva alias** |
| En el `SELECT` | Tiene que devolver **un solo valor** por fila |
| En el `WHERE` / `HAVING` | Compara contra un valor (`>`), una lista (`IN`) o pregunta si hay filas (`EXISTS`) |
| En el `ORDER BY` | Ordena por un valor calculado |

### Subselect en el FROM
Resuelvo primero la consulta de adentro y después trabajo sobre su resultado.

> **Ojo con la performance:** al resolver la subconsulta los datos se llevan a memoria y se pierden los índices, así que el motor tiene que hacer lecturas secuenciales. Por eso es "anti-performante".
>
> Nota de clase: **no se va a permitir usar `SELECT` en el `FROM`** en los ejercicios. Casi siempre se puede reemplazar con `JOIN` + `GROUP BY` (segunda consulta de abajo).

In [ ]:
-- clientes que facturaron 10000 o mas en total
SELECT * 
FROM 
        ( SELECT 
              fact_cliente  AS clie, 
              SUM(fact_total) AS suma 
       FROM Factura 
       GROUP BY fact_cliente
     ) AS t
WHERE t.suma >= 10000

In [ ]:
-- lo mismo con la razon social, resuelto SIN subselect en el FROM: JOIN + GROUP BY + HAVING
SELECT 
    fact_cliente  AS clie, 
    clie_razon_social,
    SUM(fact_total) AS suma
FROM Factura f 
JOIN Cliente c ON f.fact_cliente = c.clie_codigo
GROUP BY fact_cliente, clie_razon_social
HAVING SUM(fact_total) >= 10000

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

### Subselect en el SELECT y en el WHERE: estáticos y dinámicos
- **Estático (no correlacionado):** no usa nada de la consulta de afuera. Se ejecuta **una sola vez**.
- **Dinámico (correlacionado):** usa una columna de la consulta de afuera (acá `clie_codigo`). Se ejecuta **una vez por cada fila** de afuera, por eso la condición tiene que tener un índice.

Dos operadores para el `WHERE`:
- `IN ( subselect )`: el valor está dentro de la lista que devuelve el subselect
- `EXISTS ( subselect )`: hay al menos una fila en el subselect. No importa qué columnas devuelve, por eso se escribe `SELECT 1`

In [ ]:
-- ESTATICOS: la consulta de adentro no depende del cliente
SELECT 
    clie_codigo, 
    (SELECT COUNT(*) FROM Factura) AS total_facturas             -- da lo mismo en todas las filas
FROM Cliente 
WHERE clie_codigo IN ( SELECT fact_cliente FROM Factura )        -- solo clientes que aparecen en Factura

In [ ]:
-- DINAMICOS: la consulta de adentro usa clie_codigo de la consulta de afuera
SELECT 
    clie_codigo, 
    (SELECT COUNT(*) FROM Factura WHERE fact_cliente = clie_codigo) AS total_facturas    -- cambia con cada cliente
FROM Cliente 
WHERE EXISTS ( SELECT 1 FROM Factura WHERE fact_cliente = clie_codigo )                  -- solo clientes con alguna factura

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

También se puede ordenar por un subselect (hay que repetirlo, porque `ORDER BY` de esta forma no usa el alias).

In [ ]:
SELECT 
    clie_codigo, 
    (SELECT SUM(fact_total) FROM Factura WHERE fact_cliente = clie_codigo) AS total_facturado
FROM Cliente 
WHERE EXISTS ( SELECT 1 FROM Factura WHERE fact_cliente = clie_codigo )
ORDER BY (SELECT SUM(fact_total) FROM Factura WHERE fact_cliente = clie_codigo) DESC

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

### Cuidados al usar subselects y joins
- **NULL en cuentas:** `numero + NULL = NULL` y `numero > NULL` nunca es verdadero. Si un subselect puede devolver NULL (por ejemplo porque no hay filas en 2011), se envuelve con `ISNULL(subselect, 0)` para comparar siempre entre números.
- **Cardinalidad:** cada `JOIN` puede multiplicar filas (un producto con 3 componentes y 2 depósitos aparece 6 veces). Si después hago `SUM` o `AVG` sobre esas filas, el resultado sale mal. Una salida es calcular el valor con un subselect correlacionado en vez de un join. Ejemplo resuelto en `EjerciciosGuia` (ejercicio N4).

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

## Ejercicios para practicar
Con subselects (en `FROM`, `SELECT`, `WHERE`/`HAVING`), `IN`, `EXISTS`, correlacionados y los cuidados de `NULL` y cardinalidad se pueden hacer estos de la práctica:

**Ya resueltos en `EjerciciosGuia` (repaso):** Ej. 4, Ej. 5, Ej. 6 y Ej. 8 (extra A).

**Resueltos en `EjerciciosGuia` (sección *Respuestas > Clase 4*)** — probalos primero por tu cuenta:
- **Ej. 7:** por artículo: código, detalle, mayor precio, menor precio y % de diferencia respecto del menor. Solo los que tienen stock.
- **Ej. 12:** por producto con operaciones en 2012: nombre, cantidad de clientes distintos que lo compraron, importe promedio pagado, cantidad de depósitos con stock y stock total; ordenado por monto vendido de mayor a menor. *(Ojo con la cardinalidad.)*
- **Ej. 14:** estadística de ventas por cliente en el último año (cantidad de compras, promedio, productos diferentes, mayor compra), todos los clientes, ordenados por cantidad de compras y **sin NULLs**.
- **Ej. 17:** estadística por año/mes y producto (`YYYYMM`, código, detalle, cantidad vendida, cantidad del mismo mes del año anterior, cantidad de facturas), sin NULLs.
- **Ej. 20:** ranking de los 3 mejores empleados de 2012 con puntaje 2011 y 2012 (regla de 50 facturas con `CASE`, subordinados directos).
- **Ej. 21:** por año con facturas, cantidad de clientes y de facturas "incorrectas" (total menos impuestos difiere en más de $1 de la suma de los ítems).
- **Ej. 22:** estadística por rubro y trimestre (facturas con al menos un producto del rubro, productos diferentes del rubro), sin productos compuestos y solo si superan las 100 facturas.
- **Ej. 24:** productos con composición facturados en al menos 5 facturas de los dos vendedores con mayor comisión: código, nombre y unidades facturadas.
- **Ej. 29:** estadística por producto de 2011, solo de familias con más de 20 productos: cantidad vendida, cantidad de facturas y monto total.
- **Ej. 32:** pares de familias con productos facturados juntos más de 10 veces: códigos, detalles, cantidad de facturas y total vendido.
- **Ej. 33:** venta de los componentes del producto más vendido de 2012: código, nombre, unidades, facturas, precio promedio y total facturado.
- **Ej. 34:** por rubro y mes de 2011, cantidad de facturas "mal hechas" (con productos de dos rubros distintos), devolviendo 0 cuando no hay.
- **Ej. 35:** estadística por año y producto (facturas, vendedores distintos, a cuántos productos compone, % sobre la venta total del año).